# Sprint 11 · Sesión única · Proyecto práctico BI con Power BI + Tableau (70 min) — Versión estudiante

**Proyecto:** Andes Capital Real Estate · Dashboard comercial y financiero  
**Herramientas:** Power BI Desktop + Tableau Desktop/Public  
**Dataset:** modelo estrella con 4 CSV: ventas, clientes, propiedades y calendario

> En esta sesión construirás el mismo análisis en Power BI y Tableau. La meta no es memorizar botones, sino entender cómo un dataset relacional se convierte en un dashboard ejecutivo con KPIs, filtros, medidas y narrativa de negocio.

## Cómo usar esta versión estudiante

Este notebook está preparado para que tomes apuntes, completes ejercicios y documentes tus decisiones durante la clase.

**Modo de trabajo recomendado:**

1. Lee primero la consigna de cada bloque o ejercicio.
2. Completa los espacios de respuesta antes de comparar con la explicación del instructor/a.
3. Ejecuta las celdas de setup cuando existan.
4. Escribe interpretación, dudas y decisiones en los espacios de apuntes.
5. Al final, revisa si tus respuestas conectan datos, método e implicación de negocio.

> Las salidas de ejecución fueron limpiadas para que puedas correr el notebook desde cero.


## Notas de clase principales

- El modelo estrella separa una tabla de hechos y tablas de dimensiones.
- Las relaciones correctas son esenciales para que los KPIs no se dupliquen ni se filtren mal.
- Power BI usa medidas DAX; Tableau usa campos calculados y lógica de visualización diferente.
- Comparar herramientas ayuda a entender conceptos de BI más allá de una interfaz específica.
- El mini-reporte debe explicar resultados, no solo describir gráficos.

### Mis notas iniciales

- 
- 
- 


## Fecha

> Completa aquí la fecha de la sesión.

## Objetivos de la sesión

Al finalizar esta clase, podrás:

1. Identificar la diferencia entre una **tabla de hechos** y una **tabla de dimensiones**.
2. Cargar múltiples CSV y construir un **modelo estrella** en Power BI y Tableau.
3. Crear KPIs comerciales básicos en Power BI usando **DAX**.
4. Crear campos calculados equivalentes en Tableau.
5. Construir un dashboard ejecutivo con filtros, tarjetas, barras, líneas y mapa.
6. Comparar cómo Power BI y Tableau resuelven el mismo problema analítico.
7. Redactar una recomendación ejecutiva basada en el dashboard.

## Agenda sugerida (70 minutos)

| Tiempo | Bloque | Herramienta | Resultado esperado |
|---:|---|:---:|---|
| 0–5 | Contexto del proyecto | Discusión | Entender el reto de Andes Capital |
| 5–12 | Dataset y modelo estrella | Ambas | Identificar hechos, dimensiones y relaciones |
| 12–27 | Construcción en Power BI | Power BI | Modelo + medidas DAX + primera página |
| 27–42 | Construcción en Tableau | Tableau | Relaciones + calculated fields + dashboard |
| 42–55 | Comparación y ajuste visual | Ambas | Revisar diferencias y consistencia de KPIs |
| 55–62 | Mini-reporte ejecutivo | Ambas | Traducir visuales en hallazgos |
| 62–70 | Validación, takeaways y canales de ayuda | Cierre | Confirmar aprendizaje y próximos pasos |

---
# 0 · Contexto del proyecto (5 min)

**Andes Capital Real Estate** quiere analizar el desempeño de sus ventas inmobiliarias entre 2022 y 2024.

La gerencia quiere responder:

- ¿Cuánto ingreso cerrado se generó?
- ¿Qué ciudades y tipos de propiedad generan más valor?
- ¿Qué canal comercial es más rentable?
- ¿Cómo evoluciona el ingreso por mes?
- ¿Qué segmentos de compradores concentran mayor ticket promedio?
- ¿Qué tan consistente es el resultado entre Power BI y Tableau?

La sesión se centra en un proyecto práctico: construir el mismo dashboard en ambas herramientas y comparar el flujo de trabajo.

---
# 1 · Dataset usado en la sesión (7 min)

Usaremos un **modelo estrella** con cuatro archivos CSV.

| Archivo | Rol | Descripción | Clave principal |
|---|---|---|---|
| `sprint11_hecho_ventas_propiedades.csv` | Hechos | Transacciones y oportunidades comerciales | `id_venta` |
| `sprint11_dim_clientes.csv` | Dimensión | Perfil del comprador | `id_cliente` |
| `sprint11_dim_propiedades.csv` | Dimensión | Características de la propiedad | `id_propiedad` |
| `sprint11_dim_fecha.csv` | Dimensión | Calendario para análisis temporal | `fecha` |

Relaciones esperadas:

```text
sprint11_dim_clientes[id_cliente]       1 ─── * sprint11_hecho_ventas_propiedades[id_cliente]
sprint11_dim_propiedades[id_propiedad]  1 ─── * sprint11_hecho_ventas_propiedades[id_propiedad]
sprint11_dim_fecha[fecha]               1 ─── * sprint11_hecho_ventas_propiedades[fecha_venta]
```

**Idea clave:** la tabla de hechos responde "qué ocurrió"; las dimensiones explican "quién", "qué", "dónde" y "cuándo".

In [ ]:
# Celda opcional para revisar el dataset en Google Colab antes de abrir Power BI/Tableau.
# Si subes los CSV al entorno de Colab, esta celda permite validar nombres de columnas y filas.

import pandas as pd
from pathlib import Path

DATA_DIR = Path("sprint11_dataset_andes_capital_bi")

archivos = [
    "sprint11_hecho_ventas_propiedades.csv",
    "sprint11_dim_clientes.csv",
    "sprint11_dim_propiedades.csv",
    "sprint11_dim_fecha.csv"
]

for archivo in archivos:
    ruta = DATA_DIR / archivo
    if ruta.exists():
        df = pd.read_csv(ruta)
        print(f"{archivo}: {df.shape[0]:,} filas · {df.shape[1]} columnas")
        display(df.head(3))
    else:
        print(f"No encontrado en Colab: {ruta}. Puedes omitir esta celda si trabajarás directamente en Power BI/Tableau.")


---
# 2 · Construcción en Power BI (15 min)

## 2.1 · Cargar los archivos

1. Abre **Power BI Desktop**.
2. Ve a `Inicio` → `Obtener datos` → `Texto/CSV`.
3. Carga los cuatro archivos CSV.
4. En **Transformar datos**, verifica tipos:
   - `fecha_venta` y `fecha` como fecha.
   - `precio_venta`, `comision_usd`, `descuento_pct`, `precio_m2` como número decimal.
   - IDs como texto.
5. Cierra y aplica.

## 2.2 · Crear relaciones

En la vista **Modelo**, crea o valida estas relaciones:

| Tabla origen | Campo | Tabla destino | Campo | Cardinalidad |
|---|---|---|---|---|
| `sprint11_dim_clientes` | `id_cliente` | `sprint11_hecho_ventas_propiedades` | `id_cliente` | 1 a muchos |
| `sprint11_dim_propiedades` | `id_propiedad` | `sprint11_hecho_ventas_propiedades` | `id_propiedad` | 1 a muchos |
| `sprint11_dim_fecha` | `fecha` | `sprint11_hecho_ventas_propiedades` | `fecha_venta` | 1 a muchos |

Después marca `sprint11_dim_fecha` como tabla de fechas:

`Vista de tabla` → selecciona `sprint11_dim_fecha` → `Herramientas de tabla` → `Marcar como tabla de fechas` → campo `fecha`.

## 2.3 · Medidas DAX mínimas

Crea estas medidas en Power BI. Puedes crear una tabla o carpeta llamada `_Medidas` para mantenerlas organizadas.

```dax
Ingreso Cerrado =
CALCULATE(
    SUM(sprint11_hecho_ventas_propiedades[precio_venta]),
    sprint11_hecho_ventas_propiedades[estado_venta] = "Cerrada"
)

Ventas Cerradas =
CALCULATE(
    COUNTROWS(sprint11_hecho_ventas_propiedades),
    sprint11_hecho_ventas_propiedades[estado_venta] = "Cerrada"
)

Ticket Promedio =
DIVIDE([Ingreso Cerrado], [Ventas Cerradas])

Comisión Total =
CALCULATE(
    SUM(sprint11_hecho_ventas_propiedades[comision_usd]),
    sprint11_hecho_ventas_propiedades[estado_venta] = "Cerrada"
)

Descuento Promedio =
AVERAGE(sprint11_hecho_ventas_propiedades[descuento_pct])

Precio Promedio m2 =
AVERAGE(sprint11_hecho_ventas_propiedades[precio_m2])

Ingreso YTD =
TOTALYTD(
    [Ingreso Cerrado],
    sprint11_dim_fecha[fecha]
)
```

**Validación rápida:** si filtras por ciudad, tipo de propiedad o canal, las tarjetas deben cambiar. Si no cambian, probablemente hay una relación mal configurada.

## 2.4 · Página de dashboard en Power BI

Construye una página llamada `Overview Ejecutivo`.

| Visual | Campos sugeridos | Pregunta que responde |
|---|---|---|
| Tarjeta | `Ingreso Cerrado` | ¿Cuánto se vendió? |
| Tarjeta | `Ventas Cerradas` | ¿Cuántas ventas se cerraron? |
| Tarjeta | `Ticket Promedio` | ¿Cuál es el valor promedio por venta? |
| Tarjeta | `Comisión Total` | ¿Cuánto se pagó en comisiones? |
| Línea | Eje: `sprint11_dim_fecha[anio_mes]`; valor: `Ingreso Cerrado` | ¿Cómo evoluciona el ingreso? |
| Barras | Eje: `ciudad_propiedad`; valor: `Ingreso Cerrado` | ¿Dónde se genera más valor? |
| Barras | Eje: `tipo_propiedad`; valor: `Ingreso Cerrado` | ¿Qué tipo de propiedad vende más? |
| Dona o barras | `canal_venta`; valor: `Ingreso Cerrado` | ¿Qué canal aporta más? |
| Segmentadores | Año, ciudad, segmento comprador | ¿Cómo exploro el dashboard? |

Criterio de calidad: el dashboard debe permitir leer los KPIs principales en menos de 30 segundos.

---
# 3 · Construcción en Tableau (15 min)

## 3.1 · Conectar los CSV

1. Abre **Tableau Desktop** o **Tableau Public**.
2. En `Conectar`, selecciona `Archivo de texto`.
3. Abre `sprint11_hecho_ventas_propiedades.csv`.
4. Arrastra al lienzo:
   - `sprint11_dim_clientes.csv`
   - `sprint11_dim_propiedades.csv`
   - `sprint11_dim_fecha.csv`
5. Crea relaciones lógicas:
   - `id_cliente` ↔ `id_cliente`
   - `id_propiedad` ↔ `id_propiedad`
   - `fecha_venta` ↔ `fecha`

**Diferencia clave:** Power BI trabaja con un modelo tabular y medidas DAX; Tableau prioriza relaciones lógicas y campos calculados dentro de vistas.

## 3.2 · Campos calculados en Tableau

Crea estos campos calculados.

**Ingreso Cerrado**

```tableau
SUM(
    IF [estado_venta] = "Cerrada" THEN [precio_venta] END
)
```

**Ventas Cerradas**

```tableau
COUNTD(
    IF [estado_venta] = "Cerrada" THEN [id_venta] END
)
```

**Ticket Promedio**

```tableau
[Ingreso Cerrado] / [Ventas Cerradas]
```

**Comisión Total**

```tableau
SUM(
    IF [estado_venta] = "Cerrada" THEN [comision_usd] END
)
```

**Ingreso por Cliente — LOD**

```tableau
{ FIXED [id_cliente] :
    SUM(IF [estado_venta] = "Cerrada" THEN [precio_venta] END)
}
```

El campo LOD permite calcular a nivel de cliente aunque la visualización esté agregada por ciudad, canal o tipo de propiedad.

## 3.3 · Dashboard en Tableau

Crea hojas y luego intégralas en un dashboard llamado `Overview Ejecutivo`.

| Hoja | Tipo de visual | Campos sugeridos |
|---|---|---|
| `KPI Ingreso` | Texto grande | `Ingreso Cerrado` |
| `KPI Ventas` | Texto grande | `Ventas Cerradas` |
| `Tendencia mensual` | Línea | `MONTH(fecha_venta)`, `Ingreso Cerrado` |
| `Ingreso por ciudad` | Barras | `ciudad_propiedad`, `Ingreso Cerrado` |
| `Ingreso por tipo` | Barras | `tipo_propiedad`, `Ingreso Cerrado` |
| `Ingreso por canal` | Barras o dona | `canal_venta`, `Ingreso Cerrado` |
| `Mapa` | Mapa si Tableau reconoce ciudades | `ciudad_propiedad`, `Ingreso Cerrado` |

Agrega filtros visibles:

- `anio`
- `ciudad_propiedad`
- `segmento_comprador`
- `tipo_propiedad`

Criterio de calidad: el resultado de `Ingreso Cerrado` debe coincidir razonablemente con Power BI. Si no coincide, revisa filtros, relaciones y agregaciones.

---
# 4 · Comparación Power BI vs Tableau (13 min)

Completa esta tabla durante la clase.

| Criterio | Power BI | Tableau |
|---|---|---|
| Carga de datos |  |  |
| Modelado de relaciones |  |  |
| Creación de KPIs |  |  |
| Facilidad para filtros |  |  |
| Diseño visual |  |  |
| Curva de aprendizaje |  |  |
| Mejor caso de uso |  |  |

Preguntas guía:

1. ¿En cuál herramienta fue más claro construir el modelo?
2. ¿En cuál herramienta fue más rápido crear visualizaciones?
3. ¿Dónde fue más explícito el cálculo de KPIs?
4. ¿Qué herramienta usarías para presentar a gerencia y por qué?

---
# 5 · Mini-reporte ejecutivo (7 min)

Redacta tres hallazgos y una recomendación. Usa esta estructura:

```text
Situación:
Andes Capital necesita entender el desempeño comercial de sus ventas inmobiliarias entre 2022 y 2024.

Hallazgo 1:
El mayor ingreso cerrado se concentra en ________, especialmente en ________.

Hallazgo 2:
El canal ________ genera alto ingreso, pero debe revisarse frente a su comisión total.

Hallazgo 3:
El segmento ________ tiene un ticket promedio superior/inferior al resto.

Recomendación:
Priorizar ________, monitorear ________ y profundizar el análisis de ________ antes de tomar decisiones comerciales.
```

No basta con describir el gráfico. El objetivo es conectar visualización → hallazgo → decisión.

### Espacio de trabajo del estudiante

**Mi interpretación de la consigna:**  

**Mi procedimiento / fórmula / consulta:**  

**Resultado obtenido:**  

**Conclusión breve:**  


---
# 6 · Preguntas de validación de conocimiento (5 min)

Responde individualmente antes del cierre.

1. ¿Qué diferencia hay entre una tabla de hechos y una tabla de dimensiones?
2. ¿Por qué `dim_fecha` es importante en Power BI?
3. ¿Qué problema puede ocurrir si una relación está mal configurada?
4. ¿Qué diferencia práctica viste entre una medida DAX y un calculated field de Tableau?
5. ¿Por qué conviene construir el mismo dashboard en dos herramientas durante la clase?

## Mis respuestas

> Escribe aquí tus respuestas antes de la discusión en clase.

1. 
2. 
3. 

---
# Takeaways

- Un dashboard BI confiable empieza por un **modelo de datos correcto**, no por los colores.
- La tabla de hechos registra transacciones; las dimensiones permiten segmentarlas.
- Power BI exige mayor atención al modelo, relaciones y medidas DAX.
- Tableau facilita exploración visual, pero requiere controlar bien relaciones y agregaciones.
- Un KPI debe tener una definición explícita: qué cuenta, qué excluye y bajo qué filtros se evalúa.
- La comparación entre Power BI y Tableau ayuda a separar la lógica de negocio de la interfaz de la herramienta.

### Espacio de trabajo del estudiante

**Mi interpretación de la consigna:**  

**Mi procedimiento / fórmula / consulta:**  

**Resultado obtenido:**  

**Conclusión breve:**  


---
# Canales de ayuda y próximos pasos

Usa estos canales cuando te bloquees:

1. **Notebook de clase:** revisa primero las instrucciones, relaciones y fórmulas sugeridas.
2. **Compañeros:** compara si tus KPIs coinciden con los de otra persona.
3. **Instructor o tutor:** comparte captura del modelo, medida/campo calculado y visual afectado.
4. **Documentación oficial:** consulta funciones DAX, relaciones en Power BI y calculated fields/LOD en Tableau.

Para continuar practicando:

- Agrega una segunda página enfocada en `Análisis Comercial`.
- Crea una visual de ranking Top 10 propiedades por ingreso.
- Compara comisión total vs ingreso cerrado por canal.
- Publica o exporta tu dashboard y redacta un resumen ejecutivo de una página.